# pepbenchmark.analyze Tutorial

This notebook demonstrates how to use the `pepbenchmark.analyze` package for multi-level peptide analysis.

It covers:

- FASTA-level peptide property analysis
- Amino-acid level k-mer analysis
- Sequence-level k-mer analysis
- SMILES-level descriptor analysis
- Result merging and export

## 1. Import the analysis modules and configure paths

This section imports the main analyzers, helper utilities, and common notebook dependencies. It also creates an output folder for saving tutorial artifacts.

In [1]:
from pathlib import Path

import pandas as pd

import pepbenchmark
from pepbenchmark.analyze import (
    AcidLevelAnalyzer,
    KmerAnalyzer,
    PeptidePropertiesAnalyse,
    SmilesAnalyse,
    compute_peptide_properties,
)
from pepbenchmark.analyze import utils

project_root = Path(pepbenchmark.__file__).resolve().parents[2]
output_dir = project_root / "notebook" / "analyse" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Output directory: {output_dir}")

Project root: /home/dataset-assist-0/jiahui/pepbenchmark/final/pepbenchmark
Output directory: /home/dataset-assist-0/jiahui/pepbenchmark/final/pepbenchmark/notebook/analyse/outputs


## 2. Load example sequence data and run basic checks

This section creates a small example dataset with peptide sequences, labels, and aligned example SMILES strings.

The current `SmilesAnalyse` module expects existing SMILES strings as input, so this tutorial uses a manually aligned SMILES column for demonstration.

In [2]:
data = pd.DataFrame(
    {
        "sample_id": ["pep_1", "pep_2", "pep_3", "pep_4"],
        "sequence": ["ACDEFGHIK", "LMNPQRSTV", "ACDACDACD", "MNPQRSTVWY"],
        "label": [1, 0, 1, 0],
        "smiles": ["CCO", "CCN", "CC(=O)O", "c1ccccc1O"],
    }
)

data["length"] = data["sequence"].str.len()
data["has_missing"] = data[["sequence", "smiles"]].isna().any(axis=1)
data["is_duplicate_sequence"] = data["sequence"].duplicated(keep=False)
data["natural_sequence"] = data["sequence"].map(utils.filter_to_natural)
data["contains_non_canonical"] = data["sequence"] != data["natural_sequence"]

print(data)
print()
print("Length summary:")
print(data["length"].describe())

  sample_id    sequence  label     smiles  length  has_missing  \
0     pep_1   ACDEFGHIK      1        CCO       9        False   
1     pep_2   LMNPQRSTV      0        CCN       9        False   
2     pep_3   ACDACDACD      1    CC(=O)O       9        False   
3     pep_4  MNPQRSTVWY      0  c1ccccc1O      10        False   

   is_duplicate_sequence natural_sequence  contains_non_canonical  
0                  False        ACDEFGHIK                   False  
1                  False        LMNPQRSTV                   False  
2                  False        ACDACDACD                   False  
3                  False       MNPQRSTVWY                   False  

Length summary:
count     4.00
mean      9.25
std       0.50
min       9.00
25%       9.00
50%       9.00
75%       9.25
max      10.00
Name: length, dtype: float64


## 3. Run FASTA-level analysis

`PeptidePropertiesAnalyse` computes physicochemical descriptors from peptide sequences and supports both tabular output and dataset comparison.

In [3]:
fasta_analyzer = PeptidePropertiesAnalyse(ph=7.0)
fasta_result = fasta_analyzer.compute(data, seq_col="sequence")
fasta_df = fasta_result.to_dataframe()

print(fasta_result.summary())
fasta_df.head()

Total sequences: 4
length: mean=9.250, std=0.500
mw: mean=1057.931, std=164.524
aromaticity: mean=0.078, std=0.097
instability: mean=120.822, std=69.338
isoelectric_point: mean=6.905, std=2.664


,sequence,length,mw,aromaticity,instability,isoelectric_point,helix,turn,sheet,hydrophobicity,charge,charge_at_ph7,flexibility,aliphatic_index
0,ACDEFGHIK,9,1019.1318,0.111111,118.500000,5.321971,0.333333,0.222222,0.222222,-0.322222,-1.123920,-1.123920,0.000000,54.444444
1,LMNPQRSTV,9,1045.2138,0.000000,75.911111,9.750021,0.222222,0.333333,0.333333,-0.522222,0.760092,0.760092,0.000000,75.555556
2,ACDACDACD,9,885.9399,0.000000,219.555556,4.050028,0.333333,0.333333,0.000000,0.266667,-3.227285,-3.227285,0.000000,33.333333
3,MNPQRSTVWY,10,1281.4393,0.200000,69.320000,8.497787,0.100000,0.300000,0.400000,-1.070000,0.499346,0.499346,1.018476,29.000000


In [4]:
positive_sequences = data.loc[data["label"] == 1, "sequence"]
negative_sequences = data.loc[data["label"] == 0, "sequence"]
comparison_df = fasta_analyzer.compare(
    positive_sequences,
    negative_sequences,
    properties=["length", "hydrophobicity", "charge"],
).to_dataframe()

comparison_df

,property,ks_stat,ks_pvalue,mean_diff,std_diff,js_divergence,n_group1,n_group2
0,length,0.5,1.000000,-0.500000,-0.707107,0.464501,2,2
1,hydrophobicity,1.0,0.333333,0.768333,0.029070,0.832555,2,2
2,charge,1.0,0.333333,-2.805321,1.302929,0.832555,2,2


## 4. Run amino-acid level analysis

`AcidLevelAnalyzer` computes residue-level k-mer statistics, including occurrence counts, per-sequence coverage ratios, and sequence membership.

In [5]:
acid_analyzer = AcidLevelAnalyzer(data["sequence"], k=2)
acid_stats = acid_analyzer.compute_stats()
acid_df = acid_stats.to_dataframe()

print(acid_stats.summary(topn=8))
acid_df.head(10)

k=2, unique_kmers=19, total_kmers=33
n_sequences_total=4, n_sequences_eligible=4
Top kmers: AC:4, CD:4, MN:2, NP:2, PQ:2, QR:2, RS:2, ST:2


,kmer,freq_ratio,count,occurrence
0,AC,0.50,2,4
1,CD,0.50,2,4
2,MN,0.50,2,2
3,NP,0.50,2,2
4,PQ,0.50,2,2
5,QR,0.50,2,2
6,RS,0.50,2,2
7,ST,0.50,2,2
8,TV,0.50,2,2
9,DA,0.25,1,2


In [6]:
acid_top = pd.DataFrame(
    {
        "top_by_count": list(AcidLevelAnalyzer.top_kmers(acid_stats, n=5, by="count").items()),
        "top_by_occurrence": list(AcidLevelAnalyzer.top_kmers(acid_stats, n=5, by="occurrence").items()),
        "top_by_freq": list(AcidLevelAnalyzer.top_kmers(acid_stats, n=5, by="freq").items()),
    }
)
acid_top

,top_by_count,top_by_occurrence,top_by_freq
0,"(CD, 2)","(AC, 4)","(AC, 0.5)"
1,"(AC, 2)","(CD, 4)","(CD, 0.5)"
2,"(PQ, 2)","(MN, 2)","(MN, 0.5)"
3,"(QR, 2)","(NP, 2)","(NP, 0.5)"
4,"(NP, 2)","(PQ, 2)","(PQ, 0.5)"


## 5. Run k-mer level analysis

`KmerAnalyzer` offers the same statistical workflow in a reusable sequence-level module. It is useful when comparing different `k` settings or preparing tabular outputs for modeling.

In [7]:
kmer_summaries = []
for k in [2, 3]:
    analyzer = KmerAnalyzer(data["sequence"], k=k)
    stats = analyzer.compute_stats()
    frame = stats.to_dataframe()
    kmer_summaries.append(
        {
            "k": k,
            "n_unique_kmers": stats.raw_unique_kmers,
            "n_total_kmers": stats.raw_total_kmers,
            "top_kmer": frame.iloc[0]["kmer"] if not frame.empty else None,
            "top_occurrence": frame.iloc[0]["occurrence"] if not frame.empty else None,
            "feature_rows": len(frame),
        }
    )

pd.DataFrame(kmer_summaries)

,k,n_unique_kmers,n_total_kmers,top_kmer,top_occurrence,feature_rows
0,2,19,33,AC,4,19
1,3,18,29,ACD,4,18


In [8]:
kmer_3_stats = KmerAnalyzer(data["sequence"], k=3).compute_stats()
kmer_3_df = kmer_3_stats.to_dataframe()
kmer_3_df.head(10)

,kmer,freq_ratio,count,occurrence
0,ACD,0.50,2,4
1,MNP,0.50,2,2
2,NPQ,0.50,2,2
3,PQR,0.50,2,2
4,QRS,0.50,2,2
5,RST,0.50,2,2
6,STV,0.50,2,2
7,CDA,0.25,1,2
8,DAC,0.25,1,2
9,CDE,0.25,1,1


## 6. Generate and inspect SMILES representations

The current `SmilesAnalyse` class analyzes existing SMILES strings rather than converting peptide sequences directly. In a real pipeline, these strings may come from preprocessing or an external conversion step. Here they are aligned with the peptide samples for demonstration.

In [9]:
smiles_analyzer = SmilesAnalyse(data["smiles"].tolist())
smiles_result = smiles_analyzer.compute_metrics()
smiles_df = smiles_result.to_dataframe()

print(smiles_result.summary())
smiles_df

Total molecules: 4
Property mean/std preview:
                 mean        std
mol_weight  61.329750  22.899465
logP         0.361675   0.689076
num_hbd      1.000000   0.000000
num_hba      1.000000   0.000000
tpsa        25.945000   8.047031


,smiles,mol_weight,logP,num_hbd,num_hba,tpsa,num_rings,num_rotatable_bonds,fraction_csp3,num_atoms,num_atoms_h
0,CCO,46.069,-0.0014,1,1,20.23,0,0,1.0,3,9
1,CCN,45.085,-0.0350,1,1,26.02,0,0,1.0,3,10
2,CC(=O)O,60.052,0.0909,1,1,37.30,0,0,0.5,4,8
3,c1ccccc1O,94.113,1.3922,1,1,20.23,1,0,0.0,7,13


In [10]:
atom_frequency_df = smiles_analyzer.atom_frequency(include_h=False, normalize=True)
atom_frequency_df

,smiles,C,O,N
0,CCO,0.666667,0.333333,0.000000
1,CCN,0.666667,0.000000,0.333333
2,CC(=O)O,0.500000,0.500000,0.000000
3,c1ccccc1O,0.857143,0.142857,0.000000


## 7. Summarize multi-level analysis results

The outputs from sequence properties, k-mer summaries, and SMILES descriptors can be aligned by sample index and merged into a single feature table.

In [11]:
feature_table = (
    data[["sample_id", "sequence", "label", "smiles"]]
    .join(fasta_df.drop(columns=["sequence"]).add_prefix("fasta_"))
    .join(smiles_df.drop(columns=["smiles"]).add_prefix("smiles_"))
)

feature_table.head()

,sample_id,sequence,label,smiles,fasta_length,fasta_mw,fasta_aromaticity,fasta_instability,fasta_isoelectric_point,fasta_helix,...,smiles_mol_weight,smiles_logP,smiles_num_hbd,smiles_num_hba,smiles_tpsa,smiles_num_rings,smiles_num_rotatable_bonds,smiles_fraction_csp3,smiles_num_atoms,smiles_num_atoms_h
0,pep_1,ACDEFGHIK,1,CCO,9,1019.1318,0.111111,118.500000,5.321971,0.333333,...,46.069,-0.0014,1,1,20.23,0,0,1.0,3,9
1,pep_2,LMNPQRSTV,0,CCN,9,1045.2138,0.000000,75.911111,9.750021,0.222222,...,45.085,-0.0350,1,1,26.02,0,0,1.0,3,10
2,pep_3,ACDACDACD,1,CC(=O)O,9,885.9399,0.000000,219.555556,4.050028,0.333333,...,60.052,0.0909,1,1,37.30,0,0,0.5,4,8
3,pep_4,MNPQRSTVWY,0,c1ccccc1O,10,1281.4393,0.200000,69.320000,8.497787,0.100000,...,94.113,1.3922,1,1,20.23,1,0,0.0,7,13


In [12]:
kmer_summary_export = pd.DataFrame(kmer_summaries)
acid_export = acid_df.head(20)

print("Feature table shape:", feature_table.shape)
print("k-mer summary shape:", kmer_summary_export.shape)
print("Acid-level preview shape:", acid_export.shape)

Feature table shape: (4, 27)
k-mer summary shape: (2, 6)
Acid-level preview shape: (19, 4)


## 8. Save analysis outputs

This section writes the main outputs to disk so they can be reused in downstream benchmarks or reports.

In [13]:
feature_path = output_dir / "feature_table.csv"
comparison_path = output_dir / "property_comparison.csv"
kmer_path = output_dir / "kmer_summary.csv"
acid_path = output_dir / "acid_level_preview.csv"
smiles_path = output_dir / "smiles_descriptors.csv"

feature_table.to_csv(feature_path, index=False)
comparison_df.to_csv(comparison_path, index=False)
kmer_summary_export.to_csv(kmer_path, index=False)
acid_export.to_csv(acid_path, index=False)
smiles_df.to_csv(smiles_path, index=False)

saved_files = pd.DataFrame(
    {
        "path": [feature_path, comparison_path, kmer_path, acid_path, smiles_path],
        "exists": [path.exists() for path in [feature_path, comparison_path, kmer_path, acid_path, smiles_path]],
        "size_bytes": [path.stat().st_size if path.exists() else 0 for path in [feature_path, comparison_path, kmer_path, acid_path, smiles_path]],
    }
)

saved_files

,path,exists,size_bytes
0,/home/dataset-assist-0/jiahui/pepbenchmark/fin...,True,1489
1,/home/dataset-assist-0/jiahui/pepbenchmark/fin...,True,332
2,/home/dataset-assist-0/jiahui/pepbenchmark/fin...,True,101
3,/home/dataset-assist-0/jiahui/pepbenchmark/fin...,True,252
4,/home/dataset-assist-0/jiahui/pepbenchmark/fin...,True,358
